<a href="https://colab.research.google.com/github/JoseLuisMirandaRuiz/SIS310-Pronostico-IA-PMP/blob/main/PROYECTO_SIS310_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "gradio>=5,<6" plotly openpyxl scikit-learn pandas numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.


In [2]:
# =====================================================================
# CELDA 1 (ejecutar primero en Colab):
#   !pip install -q "gradio>=5,<6" plotly openpyxl scikit-learn pandas numpy
#
# CELDA 2: copiar TODO lo que sigue a partir de esta línea.
# =====================================================================

import math
import os
import tempfile

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# PALETA
# ---------------------------------------------------------------------
AZUL = "#2563EB"
VERDE = "#10B981"
GRIS = "#64748B"
AMBAR = "#F59E0B"
ROJO = "#EF4444"
BLANCO = "#FFFFFF"
FONDO = "#F8FAFC"
CELESTE = "#DCEEFB"      # fondo general celeste claro
BORDE = "#9CC3EA"        # borde de tarjetas/bloques
TEXTO = "#0F172A"

COLUMNAS = ["Periodo", "Mes_Num", "Mes", "Demanda_Real", "Pedidos"]
N_TEST = 12


# ---------------------------------------------------------------------
# 1. CARGA Y VALIDACIÓN DEL CSV
# ---------------------------------------------------------------------
def cargar_datos(archivo):
    if archivo is None:
        raise gr.Error("Sube primero el archivo .csv con el historial de demanda.")
    ruta = archivo if isinstance(archivo, str) else getattr(archivo, "name", str(archivo))
    try:
        df = pd.read_csv(ruta)
    except Exception:
        df = pd.read_csv(ruta, sep=";")
    df.columns = [c.strip() for c in df.columns]
    faltan = [c for c in COLUMNAS if c not in df.columns]
    if faltan:
        raise gr.Error(f"Faltan columnas en el CSV: {', '.join(faltan)}. "
                       f"Se esperan: {', '.join(COLUMNAS)}.")
    for c in ["Periodo", "Mes_Num", "Demanda_Real", "Pedidos"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["Periodo", "Mes_Num", "Demanda_Real", "Pedidos"])
    df = df.sort_values("Periodo").reset_index(drop=True)
    if len(df) <= N_TEST + 12:
        raise gr.Error(f"El CSV tiene {len(df)} filas válidas; se necesitan más de {N_TEST + 12} "
                       "(historial para entrenar + 12 periodos de evaluación).")
    return df


# ---------------------------------------------------------------------
# 2. MÉTODO TRADICIONAL (Regresión lineal desestacionalizada)
# ---------------------------------------------------------------------
def pronostico_tradicional(train, test):
    prom_general = train["Demanda_Real"].mean()
    indice = train.groupby("Mes_Num")["Demanda_Real"].mean() / prom_general

    train = train.copy()
    train["indice_mes"] = train["Mes_Num"].map(indice)
    train["Dda_Desest"] = train["Demanda_Real"] / train["indice_mes"]

    a, b = np.polyfit(train["Periodo"], train["Dda_Desest"], 1)

    indice_test = test["Mes_Num"].map(indice).fillna(1.0)
    pron = np.round((a * test["Periodo"] + b) * indice_test, 1)
    return pron.values, float(a), float(b)


# ---------------------------------------------------------------------
# 3. INTELIGENCIA ARTIFICIAL (MLP)
# ---------------------------------------------------------------------
def construir_X(d):
    p = d["Periodo"].values.astype(float)
    m = d["Mes_Num"].values.astype(float)
    s1 = np.sin(2 * np.pi * m / 12)
    c1 = np.cos(2 * np.pi * m / 12)
    s2 = np.sin(4 * np.pi * m / 12)
    c2 = np.cos(4 * np.pi * m / 12)
    return np.column_stack([p, s1, c1, s2, c2, p * s1, p * c1, p * s2, p * c2])


def pronostico_ia(train, test):
    modelo = make_pipeline(
        StandardScaler(),
        MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", solver="lbfgs",
                     alpha=1.0, max_iter=2000, random_state=42),
    )
    modelo.fit(construir_X(train), train["Demanda_Real"].values)
    return np.round(modelo.predict(construir_X(test)), 1)


# ---------------------------------------------------------------------
# 4. PLAN MAESTRO DE PRODUCCIÓN (MPS / PMP), DPP Y COSTOS
# ---------------------------------------------------------------------
def calcular_pmp(test, pron, inv0, lote, c_alm, c_falt):
    n = len(test)
    pedidos = test["Pedidos"].values.astype(float)
    demanda = test["Demanda_Real"].values.astype(float)

    req, inv_ini, mps, inv_fin = [], [], [], []
    inv = float(inv0)
    for i in range(n):
        r = max(float(pron[i]), pedidos[i])
        if inv < r:
            q = math.ceil((r - inv) / lote) * lote
        else:
            q = 0
        fin = inv + q - r
        req.append(r)
        inv_ini.append(inv)
        mps.append(q)
        inv_fin.append(fin)
        inv = fin

    # DPP: solo en periodos con MPS > 0
    dpp = ["-"] * n
    idx_mps = [i for i in range(n) if mps[i] > 0]
    for k, i in enumerate(idx_mps):
        fin_bloque = idx_mps[k + 1] if k + 1 < len(idx_mps) else n
        dpp[i] = round(inv_ini[i] + mps[i] - pedidos[i:fin_bloque].sum(), 1)

    # Simulación real contra la Demanda_Real
    inv_real = float(inv0)
    costo_total = costo_alm = costo_falt = 0.0
    quiebres = 0
    unid_faltantes = 0.0
    inv_real_l, falt_l = [], []
    for i in range(n):
        disponible = inv_real + mps[i]
        if disponible >= demanda[i]:
            inv_real = disponible - demanda[i]
            faltante = 0.0
        else:
            faltante = demanda[i] - disponible
            inv_real = 0.0
            quiebres += 1
        costo_alm += inv_real * c_alm
        costo_falt += faltante * c_falt
        costo_total += inv_real * c_alm + faltante * c_falt
        unid_faltantes += faltante
        inv_real_l.append(inv_real)
        falt_l.append(faltante)

    tabla = pd.DataFrame({
        "Periodo": test["Periodo"].astype(int).values,
        "Mes": test["Mes"].values,
        "Pronóstico": np.round(pron, 1),
        "Pedidos": pedidos,
        "Requerimiento": np.round(req, 1),
        "Inv. Inicial": np.round(inv_ini, 1),
        "MPS / PMP": mps,
        "Inv. Final": np.round(inv_fin, 1),
        "DPP": dpp,
        "Demanda Real": demanda,
        "Inv. Real": np.round(inv_real_l, 1),
        "Faltante": np.round(falt_l, 1),
    })
    return {
        "tabla": tabla,
        "costo_total": costo_total,
        "costo_alm": costo_alm,
        "costo_falt": costo_falt,
        "quiebres": quiebres,
        "unid_faltantes": unid_faltantes,
    }


# ---------------------------------------------------------------------
# 5. MÉTRICAS
# ---------------------------------------------------------------------
def metricas(real, pron):
    real = np.asarray(real, dtype=float)
    pron = np.asarray(pron, dtype=float)
    mape = float(np.mean(np.abs(real - pron) / np.where(real == 0, np.nan, real)) * 100)
    mad = float(np.mean(np.abs(real - pron)))
    return mape, mad


# ---------------------------------------------------------------------
# 6. COMPONENTES VISUALES
# ---------------------------------------------------------------------
def tarjeta(icono, titulo, valor, detalle, bueno, color_ok):
    color = color_ok if bueno else ROJO
    return f"""
    <div style="background:{BLANCO};border:1px solid #E2E8F0;border-left:6px solid {color};
                border-radius:14px;padding:16px 18px;box-shadow:0 2px 8px rgba(15,23,42,.06);">
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
        <div style="font-size:26px;line-height:1;">{icono}</div>
        <div style="font-size:13px;font-weight:600;color:{GRIS};text-transform:uppercase;
                    letter-spacing:.04em;">{titulo}</div>
      </div>
      <div style="font-size:30px;font-weight:800;color:{color};line-height:1.15;">{valor}</div>
      <div style="font-size:12.5px;color:{GRIS};margin-top:6px;">{detalle}</div>
    </div>"""


def html_kpis(m):
    # MAPE
    red_mape_rel = (m["mape_sin"] - m["mape_ia"]) / m["mape_sin"] * 100 if m["mape_sin"] else 0
    pp = m["mape_sin"] - m["mape_ia"]
    t1 = tarjeta(
        "🎯", "Reducción del error MAPE" if pp >= 0 else "Aumento del error MAPE",
        f"{abs(red_mape_rel):.1f}%",
        f"MAPE {m['mape_sin']:.2f}% → {m['mape_ia']:.2f}% ({pp:+.2f} pp)", pp >= 0, AZUL)
    # MAD
    red_mad_rel = (m["mad_sin"] - m["mad_ia"]) / m["mad_sin"] * 100 if m["mad_sin"] else 0
    dmad = m["mad_sin"] - m["mad_ia"]
    t2 = tarjeta(
        "📉", "Reducción del error MAD" if dmad >= 0 else "Aumento del error MAD",
        f"{abs(red_mad_rel):.1f}%",
        f"MAD {m['mad_sin']:,.1f} → {m['mad_ia']:,.1f} unidades ({dmad:+,.1f})", dmad >= 0, AZUL)
    # Ahorro
    ahorro = m["costo_sin"] - m["costo_ia"]
    t3 = tarjeta(
        "💰", "Ahorro económico" if ahorro >= 0 else "Sobrecosto con IA",
        f"Bs. {abs(ahorro):,.0f}",
        f"Costo total: Bs. {m['costo_sin']:,.0f} → Bs. {m['costo_ia']:,.0f}", ahorro >= 0, VERDE)
    # Quiebres
    evitados = m["q_sin"] - m["q_ia"]
    t4 = tarjeta(
        "🛡️", "Quiebres de stock evitados" if evitados >= 0 else "Quiebres adicionales",
        f"{abs(int(evitados))}",
        f"Periodos con quiebre: {m['q_sin']} → {m['q_ia']} de {N_TEST}", evitados >= 0, VERDE)
    return f"""
    <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));
                gap:14px;margin:6px 0 14px 0;">{t1}{t2}{t3}{t4}</div>"""


def estilo_fig(fig, titulo, alto=420):
    fig.update_layout(
        title=dict(text=titulo, font=dict(size=17, color=TEXTO)),
        template="plotly_white", paper_bgcolor=BLANCO, plot_bgcolor=BLANCO,
        font=dict(family="Inter, Segoe UI, Arial", color=TEXTO),
        hovermode="x unified", height=alto, autosize=True,
        title_x=0.01, title_y=0.96, title_xanchor="left",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0),
        margin=dict(l=60, r=30, t=105, b=55),
    )
    fig.update_xaxes(gridcolor="#E2E8F0")
    fig.update_yaxes(gridcolor="#E2E8F0")
    return fig


def fig_pronosticos(train, test, p_sin, p_ia):
    """Gráfico de precisión estacional: solo los 12 meses evaluados, con nombres de mes."""
    meses = test["Mes"].astype(str).tolist()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=meses, y=test["Demanda_Real"], mode="lines+markers", name="Demanda Real",
        line=dict(color="#000000", width=3), marker=dict(size=9, symbol="circle"),
        hovertemplate="%{x}<br>Real: %{y:,.0f}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=meses, y=p_sin, mode="lines+markers", name="Tradicional (Regresión Estacional)",
        line=dict(color="#D9534F", width=2.5, dash="dash"),
        marker=dict(size=9, symbol="square"),
        hovertemplate="%{x}<br>Tradicional: %{y:,.1f}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=meses, y=p_ia, mode="lines+markers", name="Con IA (Red Neuronal)",
        line=dict(color="#0B77CC", width=3), marker=dict(size=9, symbol="triangle-up"),
        hovertemplate="%{x}<br>Con IA: %{y:,.1f}<extra></extra>"))
    estilo_fig(fig, "Gráfico comparativo de ajuste estacional: Tradicional vs. Inteligencia Artificial",
               alto=470)
    fig.update_layout(
        title_x=0.5, title_xanchor="center", title_y=0.95,
        legend=dict(orientation="v", x=0.01, y=0.99, xanchor="left", yanchor="top",
                    bgcolor="rgba(255,255,255,0.92)", bordercolor="#CBD5E1", borderwidth=1,
                    font=dict(size=13)),
        margin=dict(l=70, r=30, t=80, b=100),
    )
    fig.update_xaxes(categoryorder="array", categoryarray=meses, tickangle=-45,
                     showgrid=True, gridcolor="#E2E8F0")
    fig.update_yaxes(title="Unidades", showgrid=True, gridcolor="#E2E8F0")
    return fig


def fig_errores(test, p_sin, p_ia):
    real = test["Demanda_Real"].values
    fig = go.Figure()
    fig.add_trace(go.Bar(x=test["Mes"], y=np.abs(real - p_sin), name="Sin IA",
                         marker_color=AZUL,
                         hovertemplate="%{x}<br>Error abs. Sin IA: %{y:,.1f}<extra></extra>"))
    fig.add_trace(go.Bar(x=test["Mes"], y=np.abs(real - p_ia), name="Con IA",
                         marker_color=VERDE,
                         hovertemplate="%{x}<br>Error abs. Con IA: %{y:,.1f}<extra></extra>"))
    fig.update_layout(barmode="group")
    fig.update_yaxes(title="Error absoluto (unidades)")
    return estilo_fig(fig, "Error absoluto por periodo")


def fig_kpis(m):
    fig = make_subplots(rows=1, cols=4, subplot_titles=(
        "MAPE (%)", "MAD (unid.)", "Costo total (Bs.)", "Quiebres de stock"))
    datos = [
        (1, [m["mape_sin"], m["mape_ia"]], "%{y:.2f}"),
        (2, [m["mad_sin"], m["mad_ia"]], "%{y:,.1f}"),
        (3, [m["costo_sin"], m["costo_ia"]], "Bs. %{y:,.0f}"),
        (4, [m["q_sin"], m["q_ia"]], "%{y:.0f}"),
    ]
    for col, vals, fmt in datos:
        fig.add_trace(go.Bar(
            x=["Sin IA", "Con IA"], y=vals, marker_color=[AZUL, VERDE],
            text=[f"{v:,.2f}" if col <= 2 else f"{v:,.0f}" for v in vals],
            textposition="outside", cliponaxis=False, showlegend=False,
            textfont=dict(size=14, color=TEXTO),
            hovertemplate="%{x}: " + fmt + "<extra></extra>"), row=1, col=col)
        tope = max(vals) if max(vals) > 0 else 1
        fig.update_yaxes(range=[0, tope * 1.35], row=1, col=col)
    estilo_fig(fig, "Comparación de KPIs: Sin IA vs. Con IA", alto=440)
    fig.update_layout(hovermode="closest", showlegend=False,
                      margin=dict(l=50, r=30, t=130, b=50))
    fig.update_annotations(font=dict(size=15, color=TEXTO), yshift=8)
    return fig


def fig_pmp(res, titulo, color):
    t = res["tabla"]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=t["Mes"], y=t["MPS / PMP"], name="Producción MPS", marker_color=color, opacity=0.85,
        hovertemplate="%{x}<br>MPS: %{y:,.0f}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=t["Mes"], y=t["Demanda Real"], name="Demanda real", mode="lines+markers",
        line=dict(color=GRIS, width=3), marker=dict(size=7),
        hovertemplate="Demanda: %{y:,.0f}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=t["Mes"], y=t["Inv. Final"], name="Inventario final (plan)", mode="lines+markers",
        line=dict(color=AMBAR, width=3, dash="dot"), marker=dict(size=7, symbol="square"),
        hovertemplate="Inv. final: %{y:,.1f}<extra></extra>"))
    fig.update_yaxes(title="Unidades")
    return estilo_fig(fig, titulo)


# ---------------------------------------------------------------------
# 7. FLUJO PRINCIPAL
# ---------------------------------------------------------------------
def ejecutar(archivo, inv0, lote, c_alm, c_falt):
    df = cargar_datos(archivo)
    if lote is None or lote <= 0:
        raise gr.Error("El tamaño de lote debe ser mayor que 0.")
    inv0 = 0 if inv0 is None else inv0
    c_alm = 0 if c_alm is None else c_alm
    c_falt = 0 if c_falt is None else c_falt

    train = df.iloc[:-N_TEST].copy()
    test = df.iloc[-N_TEST:].copy().reset_index(drop=True)

    p_sin, a, b = pronostico_tradicional(train, test)
    p_ia = pronostico_ia(train, test)

    real = test["Demanda_Real"].values
    mape_sin, mad_sin = metricas(real, p_sin)
    mape_ia, mad_ia = metricas(real, p_ia)

    res_sin = calcular_pmp(test, p_sin, inv0, lote, c_alm, c_falt)
    res_ia = calcular_pmp(test, p_ia, inv0, lote, c_alm, c_falt)

    m = dict(
        mape_sin=mape_sin, mape_ia=mape_ia, mad_sin=mad_sin, mad_ia=mad_ia,
        costo_sin=res_sin["costo_total"], costo_ia=res_ia["costo_total"],
        q_sin=res_sin["quiebres"], q_ia=res_ia["quiebres"],
    )

    def mejora(sin, con, fmt):
        return fmt.format(sin - con)

    comp = pd.DataFrame({
        "Indicador": ["MAPE (%)", "MAD (unidades)", "Costo de almacenamiento (Bs.)",
                      "Costo por faltante (Bs.)", "Costo total operativo (Bs.)",
                      "Unidades faltantes", "Periodos con quiebre de stock"],
        "Sin IA (Regresión)": [
            f"{mape_sin:.2f}", f"{mad_sin:,.1f}", f"{res_sin['costo_alm']:,.0f}",
            f"{res_sin['costo_falt']:,.0f}", f"{res_sin['costo_total']:,.0f}",
            f"{res_sin['unid_faltantes']:,.0f}", f"{res_sin['quiebres']}"],
        "Con IA (MLP)": [
            f"{mape_ia:.2f}", f"{mad_ia:,.1f}", f"{res_ia['costo_alm']:,.0f}",
            f"{res_ia['costo_falt']:,.0f}", f"{res_ia['costo_total']:,.0f}",
            f"{res_ia['unid_faltantes']:,.0f}", f"{res_ia['quiebres']}"],
        "Mejora con IA": [
            mejora(mape_sin, mape_ia, "{:+.2f} pp"), mejora(mad_sin, mad_ia, "{:+,.1f}"),
            mejora(res_sin["costo_alm"], res_ia["costo_alm"], "{:+,.0f}"),
            mejora(res_sin["costo_falt"], res_ia["costo_falt"], "{:+,.0f}"),
            mejora(res_sin["costo_total"], res_ia["costo_total"], "{:+,.0f}"),
            mejora(res_sin["unid_faltantes"], res_ia["unid_faltantes"], "{:+,.0f}"),
            mejora(res_sin["quiebres"], res_ia["quiebres"], "{:+d}")],
    })

    pron_df = pd.DataFrame({
        "Periodo": test["Periodo"].astype(int), "Mes": test["Mes"],
        "Demanda_Real": real, "Pedidos": test["Pedidos"],
        "Pronostico_Sin_IA": p_sin, "Pronostico_Con_IA": p_ia})

    estado = dict(res_sin=res_sin, res_ia=res_ia, comp=comp, pron=pron_df,
                  params=dict(inv0=inv0, lote=lote, c_alm=c_alm, c_falt=c_falt))

    nota = (f"<div style='color:{GRIS};font-size:13px;margin-bottom:8px;'>"
            f"Entrenamiento: <b>{len(train)}</b> periodos · Evaluación: <b>{N_TEST}</b> periodos · "
            f"Tendencia desestacionalizada: <b>Dda = {a:,.3f}·Periodo + {b:,.2f}</b></div>")

    return (
        nota + html_kpis(m), comp,
        fig_pronosticos(train, test, p_sin, p_ia),
        fig_errores(test, p_sin, p_ia),
        fig_kpis(m),
        res_ia["tabla"], fig_pmp(res_ia, "MPS Con IA vs. Demanda e Inventario Final", VERDE),
        estado,
    )


def mostrar_pmp(escenario, estado):
    if not estado:
        return pd.DataFrame(), go.Figure()
    if escenario.startswith("Sin"):
        res, color, tit = estado["res_sin"], AZUL, "MPS Sin IA vs. Demanda e Inventario Final"
    else:
        res, color, tit = estado["res_ia"], VERDE, "MPS Con IA vs. Demanda e Inventario Final"
    return res["tabla"], fig_pmp(res, tit, color)


def construir_excel(ruta, estado):
    """Excel interactivo: fórmulas vivas, gráficos nativos y explicación de cada hoja."""
    from openpyxl import Workbook
    from openpyxl.chart import BarChart, LineChart, Reference
    from openpyxl.chart.label import DataLabelList
    from openpyxl.chart.series import DataPoint
    from openpyxl.formatting.rule import CellIsRule
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

    P = estado["params"]
    t_sin = estado["res_sin"]["tabla"]
    t_ia = estado["res_ia"]["tabla"]
    pron = estado["pron"]
    n = len(pron)
    HDR, F0 = 5, 6              # fila de encabezado y primera fila de datos
    F1 = F0 + n - 1             # última fila de datos
    AZ, VE, CE, AM, GR = "2563EB", "10B981", "DCEEFB", "FEF3C7", "F1F5F9"
    lado = Side(style="thin", color="9CC3EA")
    borde = Border(left=lado, right=lado, top=lado, bottom=lado)

    def hdr(c, color=AZ):
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor=color)
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border = borde

    def titulo(ws, texto, hasta):
        ws["A1"] = texto
        ws.merge_cells(f"A1:{hasta}1")
        ws["A1"].font = Font(size=16, bold=True, color="FFFFFF")
        ws["A1"].fill = PatternFill("solid", fgColor=AZ)
        ws["A1"].alignment = Alignment(vertical="center", indent=1)
        ws.row_dimensions[1].height = 32

    def nota(ws, fila, texto, hasta, alto=48, color=CE):
        ws[f"A{fila}"] = texto
        ws.merge_cells(f"A{fila}:{hasta}{fila}")
        c = ws[f"A{fila}"]
        c.alignment = Alignment(wrap_text=True, vertical="top", indent=1)
        c.fill = PatternFill("solid", fgColor=color)
        c.font = Font(color="0F172A", size=11)
        ws.row_dimensions[fila].height = alto

    def celda(ws, ref, valor, fmt=None, negrita=False, fondo=None):
        c = ws[ref]
        c.value = valor
        c.border = borde
        c.alignment = Alignment(vertical="center", wrap_text=True)
        if fmt:
            c.number_format = fmt
        if negrita:
            c.font = Font(bold=True)
        if fondo:
            c.fill = PatternFill("solid", fgColor=fondo)
        return c

    def estilo_ejes(ch):
        ch.x_axis.delete = False
        ch.y_axis.delete = False

    wb = Workbook()

    # ================= HOJA 1: RESUMEN =================
    ws = wb.active
    ws.title = "Resumen"
    ws.sheet_properties.tabColor = AZ
    titulo(ws, "Reporte: Pronóstico con IA y Plan Maestro de Producción (SIS-310)", "E")
    nota(ws, 3,
         "¿QUÉ SE HIZO? Se cargó el historial de demanda (36 periodos de entrenamiento + 12 de evaluación). "
         "Con él se pronosticaron los últimos 12 meses con dos métodos: el TRADICIONAL (regresión lineal "
         "desestacionalizada) y la INTELIGENCIA ARTIFICIAL (red neuronal MLP). Cada pronóstico alimentó un Plan "
         "Maestro de Producción (PMP) con lote fijo, y ambos planes se evaluaron contra la demanda real para "
         "medir errores, costos y quiebres de stock.", "E", alto=78)
    nota(ws, 4,
         "HOJAS DEL ARCHIVO →  Pronosticos: demanda real vs. pronósticos y errores  |  PMP_Con_IA y PMP_Sin_IA: "
         "plan de producción mes a mes con su gráfico  |  Explicacion: método, fórmulas y glosario. "
         "TIP: haz clic en cualquier celda con número para ver su fórmula; las celdas AMARILLAS se pueden "
         "cambiar y los costos se recalculan solos.", "E", alto=62, color=GR)

    ws["A6"] = "Parámetros usados en la ejecución"
    ws["A6"].font = Font(bold=True, size=12)
    params = [
        ("Inventario inicial (unidades)", P["inv0"], "#,##0", GR),
        ("Tamaño de lote fijo MPS (unidades)", P["lote"], "#,##0", GR),
        ("Costo de almacenamiento (Bs./unidad)", P["c_alm"], "#,##0.00", AM),
        ("Costo por faltante (Bs./unidad)", P["c_falt"], "#,##0.00", AM),
    ]
    for k, (et, val, fmt, fondo) in enumerate(params):
        celda(ws, f"A{7 + k}", et, negrita=True)
        celda(ws, f"B{7 + k}", val, fmt, fondo=fondo)
    ws["C9"] = "← editable (amarillo)"
    ws["C10"] = "← editable (amarillo)"
    ws["C7"] = "(fijo: define el plan)"
    ws["C8"] = "(fijo: define el plan)"
    for r in range(7, 11):
        ws[f"C{r}"].font = Font(italic=True, color="64748B")

    H = 12
    for col, txt in zip("ABCDE", ["Indicador (KPI)", "Sin IA (Regresión)", "Con IA (MLP)",
                                  "Mejora con IA (Sin − Con)", "¿Qué significa?"]):
        ws[f"{col}{H}"] = txt
        hdr(ws[f"{col}{H}"], VE if col == "C" else AZ)
    rp = f"{F0}:{F1}"  # rango de filas
    kpis = [
        ("MAPE (error porcentual medio)",
         f"=AVERAGE(Pronosticos!I{F0}:I{F1})", f"=AVERAGE(Pronosticos!J{F0}:J{F1})", "0.00%",
         "En qué porcentaje se equivoca el pronóstico en promedio. Menos es mejor."),
        ("MAD (desviación absoluta media, unid.)",
         f"=AVERAGE(Pronosticos!G{F0}:G{F1})", f"=AVERAGE(Pronosticos!H{F0}:H{F1})", "#,##0.0",
         "Cuántas unidades se equivoca el pronóstico en promedio cada mes."),
        ("Costo de almacenamiento (Bs.)",
         f"=SUM(PMP_Sin_IA!K{F0}:K{F1})*$B$9", f"=SUM(PMP_Con_IA!K{F0}:K{F1})*$B$9", "#,##0",
         "Unidades que sobraron al cierre de cada mes × costo de almacenar una unidad."),
        ("Costo por faltante (Bs.)",
         f"=SUM(PMP_Sin_IA!L{F0}:L{F1})*$B$10", f"=SUM(PMP_Con_IA!L{F0}:L{F1})*$B$10", "#,##0",
         "Unidades que no se pudieron entregar × costo por cada unidad faltante."),
        ("COSTO TOTAL OPERATIVO (Bs.)", "=B15+B16", "=C15+C16", "#,##0",
         "Almacenamiento + faltantes. Es el costo real de operar con cada plan."),
        ("Unidades faltantes", f"=SUM(PMP_Sin_IA!L{F0}:L{F1})", f"=SUM(PMP_Con_IA!L{F0}:L{F1})", "#,##0",
         "Demanda real que quedó sin cubrir en los 12 meses."),
        ("Periodos con quiebre de stock",
         f'=COUNTIF(PMP_Sin_IA!L{F0}:L{F1},">0")', f'=COUNTIF(PMP_Con_IA!L{F0}:L{F1},">0")', "0",
         "Meses en los que la demanda real superó lo disponible (se quedó sin producto)."),
    ]
    for k, (et, fs, fc, fmt, expl) in enumerate(kpis):
        r = H + 1 + k
        celda(ws, f"A{r}", et, negrita=True)
        celda(ws, f"B{r}", fs, fmt)
        celda(ws, f"C{r}", fc, fmt)
        celda(ws, f"D{r}", f"=B{r}-C{r}", fmt, negrita=True)
        celda(ws, f"E{r}", expl)
        ws.row_dimensions[r].height = 32
    ws["A20"] = ("Lectura de la columna «Mejora»: un valor POSITIVO significa que la IA es mejor (menos error, "
                 "menos costo o menos quiebres); un valor negativo indica que el método tradicional fue mejor en ese indicador.")
    ws.merge_cells("A20:E20")
    ws["A20"].alignment = Alignment(wrap_text=True, vertical="top")
    ws["A20"].font = Font(italic=True, color="475569")
    ws.row_dimensions[20].height = 34

    for col, ancho in zip("ABCDE", [40, 20, 20, 24, 78]):
        ws.column_dimensions[col].width = ancho

    # Gráficos de KPIs (barras Sin IA vs Con IA)
    graf = [("MAPE: error porcentual medio", H + 1, "A22"), ("MAD: desviación media (unidades)", H + 2, "E22"),
            ("Costo total operativo (Bs.)", H + 5, "A38"), ("Periodos con quiebre de stock", H + 7, "E38")]
    for tit, fila, ancla in graf:
        ch = BarChart()
        ch.type = "col"
        ch.title = tit
        ch.legend = None
        ch.add_data(Reference(ws, min_col=2, max_col=3, min_row=fila, max_row=fila), from_rows=True,
                    titles_from_data=False)
        ch.set_categories(Reference(ws, min_col=2, max_col=3, min_row=H, max_row=H))
        for idx, color in enumerate([AZ, VE]):
            pt = DataPoint(idx=idx)
            pt.graphicalProperties.solidFill = color
            ch.series[0].dPt.append(pt)
        ch.dataLabels = DataLabelList()
        ch.dataLabels.showVal = True
        ch.dataLabels.showSerName = False
        ch.dataLabels.showCatName = False
        ch.dataLabels.showLegendKey = False
        ch.dataLabels.showPercent = False
        ch.width, ch.height = 15, 7.5
        estilo_ejes(ch)
        ws.add_chart(ch, ancla)

    # ================= HOJA 2: PRONÓSTICOS =================
    wp = wb.create_sheet("Pronosticos")
    wp.sheet_properties.tabColor = "F59E0B"
    titulo(wp, "Pronósticos de los últimos 12 periodos: Sin IA vs. Con IA", "J")
    nota(wp, 2,
         "QUÉ MUESTRA: la demanda real de los 12 meses de evaluación junto con lo que predijo cada método. "
         "SIN IA = (a·Periodo + b) × índice estacional del mes (regresión lineal desestacionalizada). "
         "CON IA = red neuronal MLP (64 y 32 neuronas) entrenada con Periodo, seno/coseno del mes e interacciones.",
         "J", alto=52)
    nota(wp, 3,
         "ERRORES (fórmulas vivas): Error absoluto = |Real − Pronóstico|; % Error = Error absoluto ÷ Real. "
         "El PROMEDIO de la columna G/H es el MAD y el de I/J es el MAPE que aparecen en la hoja Resumen. "
         "Filtra o ordena con las flechas del encabezado.", "J", alto=52, color=GR)
    cols = ["Periodo", "Mes", "Demanda Real", "Pedidos", "Pronóstico Sin IA", "Pronóstico Con IA",
            "Error abs. Sin IA", "Error abs. Con IA", "% Error Sin IA", "% Error Con IA"]
    for k, c in enumerate(cols):
        cc = wp.cell(row=HDR, column=k + 1, value=c)
        hdr(cc, VE if "Con IA" in c else AZ)
    wp.row_dimensions[HDR].height = 32
    for i in range(n):
        r = F0 + i
        fila = pron.iloc[i]
        celda(wp, f"A{r}", int(fila["Periodo"]), "0")
        celda(wp, f"B{r}", str(fila["Mes"]))
        celda(wp, f"C{r}", float(fila["Demanda_Real"]), "#,##0")
        celda(wp, f"D{r}", float(fila["Pedidos"]), "#,##0")
        celda(wp, f"E{r}", float(fila["Pronostico_Sin_IA"]), "#,##0.0")
        celda(wp, f"F{r}", float(fila["Pronostico_Con_IA"]), "#,##0.0")
        celda(wp, f"G{r}", f"=ABS(C{r}-E{r})", "#,##0.0")
        celda(wp, f"H{r}", f"=ABS(C{r}-F{r})", "#,##0.0")
        celda(wp, f"I{r}", f"=G{r}/C{r}", "0.00%")
        celda(wp, f"J{r}", f"=H{r}/C{r}", "0.00%")
    rt = F1 + 1
    celda(wp, f"A{rt}", "Promedio", negrita=True, fondo=CE)
    wp.merge_cells(f"A{rt}:F{rt}")
    for col, fmt in zip("GHIJ", ["#,##0.0", "#,##0.0", "0.00%", "0.00%"]):
        celda(wp, f"{col}{rt}", f"=AVERAGE({col}{F0}:{col}{F1})", fmt, negrita=True, fondo=CE)
    wp.auto_filter.ref = f"A{HDR}:J{F1}"
    wp.freeze_panes = f"C{F0}"
    for col, ancho in zip("ABCDEFGHIJ", [9, 14, 14, 12, 16, 16, 15, 15, 14, 14]):
        wp.column_dimensions[col].width = ancho

    lc = LineChart()
    lc.title = "Ajuste estacional: demanda real vs. Tradicional vs. IA"
    lc.y_axis.title = "Unidades"
    for col, color, dash, simbolo in [(3, "000000", None, "circle"), (5, "D9534F", "dash", "square"),
                                      (6, "0B77CC", None, "triangle")]:
        lc.add_data(Reference(wp, min_col=col, min_row=HDR, max_row=F1), titles_from_data=True)
        sr = lc.series[-1]
        sr.graphicalProperties.line.solidFill = color
        sr.graphicalProperties.line.width = 28000
        if dash:
            sr.graphicalProperties.line.dashStyle = dash
        sr.marker.symbol = simbolo
        sr.marker.size = 8
        sr.marker.graphicalProperties.solidFill = color
        sr.smooth = False
    lc.set_categories(Reference(wp, min_col=2, min_row=F0, max_row=F1))
    lc.width, lc.height = 24, 10
    estilo_ejes(lc)
    nota(wp, rt + 2,
         "CÓMO LEER EL GRÁFICO: la línea negra es la realidad. La roja punteada (Tradicional) se queda por debajo en los "
         "meses pico porque usa un índice estacional fijo; la azul (IA) la sigue de cerca porque aprende que el efecto "
         "estacional cambia con el tiempo.", "J", alto=48, color=GR)
    wp.add_chart(lc, f"A{rt + 3}")

    bc = BarChart()
    bc.type = "col"
    bc.title = "Error absoluto por mes (unidades)"
    for col, color in [(7, AZ), (8, VE)]:
        bc.add_data(Reference(wp, min_col=col, min_row=HDR, max_row=F1), titles_from_data=True)
        bc.series[-1].graphicalProperties.solidFill = color
    bc.set_categories(Reference(wp, min_col=2, min_row=F0, max_row=F1))
    bc.width, bc.height = 24, 9
    estilo_ejes(bc)
    wp.add_chart(bc, f"A{rt + 25}")

    # ================= HOJAS 3 y 4: PMP =================
    def hoja_pmp(nombre, tabla, color, escenario, texto_pron):
        w = wb.create_sheet(nombre)
        w.sheet_properties.tabColor = color
        titulo(w, f"Plan Maestro de Producción (MPS / PMP) – {escenario}", "L")
        nota(w, 2,
             f"QUÉ MUESTRA: el plan de producción mes a mes usando {texto_pron}. "
             "Requerimiento = máx(Pronóstico, Pedidos). Si el Inv. Inicial no alcanza, se produce en múltiplos del lote fijo: "
             f"MPS = ⌈(Requerimiento − Inv. Inicial) ÷ {P['lote']:,.0f}⌉ × {P['lote']:,.0f}. "
             "Inv. Final = Inv. Inicial + MPS − Requerimiento.", "L", alto=64)
        nota(w, 3,
             "DPP (Disponible Para Promesa): solo en meses con producción = Inv. Inicial + MPS − pedidos hasta antes del próximo lote; "
             "es lo que se puede prometer a nuevos clientes. INV. REAL / FALTANTE: simulación contra la demanda real "
             "(si Disponible ≥ Demanda queda inventario; si no, hay faltante y quiebre). "
             "Rojo = mes con faltante · Verde = mes con producción.", "L", alto=64, color=GR)
        for k, c in enumerate(tabla.columns):
            cc = w.cell(row=HDR, column=k + 1, value=c)
            hdr(cc, color)
        w.row_dimensions[HDR].height = 32
        for i in range(n):
            r = F0 + i
            f = tabla.iloc[i]
            celda(w, f"A{r}", int(f["Periodo"]), "0")
            celda(w, f"B{r}", str(f["Mes"]))
            celda(w, f"C{r}", float(f["Pronóstico"]), "#,##0.0")
            celda(w, f"D{r}", float(f["Pedidos"]), "#,##0")
            celda(w, f"E{r}", f"=MAX(C{r},D{r})", "#,##0.0")                       # Requerimiento
            if i == 0:
                celda(w, f"F{r}", float(f["Inv. Inicial"]), "#,##0.0")             # Inventario inicial
            else:
                celda(w, f"F{r}", f"=H{r - 1}", "#,##0.0")                         # = Inv. final anterior
            celda(w, f"G{r}", float(f["MPS / PMP"]), "#,##0")                      # MPS (regla de lote)
            celda(w, f"H{r}", f"=F{r}+G{r}-E{r}", "#,##0.0")                       # Inv. Final
            dpp = f["DPP"]
            celda(w, f"I{r}", dpp if isinstance(dpp, str) else float(dpp), "#,##0.0")
            celda(w, f"J{r}", float(f["Demanda Real"]), "#,##0")
            prev = f"$F${F0}" if i == 0 else f"K{r - 1}"
            celda(w, f"K{r}", f"=MAX(0,{prev}+G{r}-J{r})", "#,##0.0")             # Inv. real
            celda(w, f"L{r}", f"=MAX(0,J{r}-({prev}+G{r}))", "#,##0.0")            # Faltante
            w[f"I{r}"].alignment = Alignment(horizontal="center")
        rt2 = F1 + 1
        celda(w, f"A{rt2}", "Totales", negrita=True, fondo=CE)
        w.merge_cells(f"A{rt2}:B{rt2}")
        for col in "CDEGJKL":
            celda(w, f"{col}{rt2}", f"=SUM({col}{F0}:{col}{F1})", "#,##0", negrita=True, fondo=CE)
        w.conditional_formatting.add(f"L{F0}:L{F1}", CellIsRule(operator="greaterThan", formula=["0"],
                                     fill=PatternFill("solid", bgColor="FECACA", fgColor="FECACA")))
        w.conditional_formatting.add(f"G{F0}:G{F1}", CellIsRule(operator="greaterThan", formula=["0"],
                                     fill=PatternFill("solid", bgColor="BBF7D0", fgColor="BBF7D0")))
        w.auto_filter.ref = f"A{HDR}:L{F1}"
        w.freeze_panes = f"C{F0}"
        for col, ancho in zip("ABCDEFGHIJKL", [9, 14, 13, 11, 15, 13, 12, 13, 12, 13, 12, 12]):
            w.column_dimensions[col].width = ancho

        b = BarChart()
        b.type = "col"
        b.title = f"MPS vs. demanda real e inventario final – {escenario}"
        b.y_axis.title = "Unidades"
        b.add_data(Reference(w, min_col=7, min_row=HDR, max_row=F1), titles_from_data=True)
        b.series[0].graphicalProperties.solidFill = color
        b.set_categories(Reference(w, min_col=2, min_row=F0, max_row=F1))
        ln = LineChart()
        for col, colr in [(10, "64748B"), (8, "F59E0B")]:
            ln.add_data(Reference(w, min_col=col, min_row=HDR, max_row=F1), titles_from_data=True)
            ln.series[-1].graphicalProperties.line.solidFill = colr
            ln.series[-1].graphicalProperties.line.width = 28000
            ln.series[-1].marker.symbol = "circle"
            ln.series[-1].marker.graphicalProperties.solidFill = colr
            ln.series[-1].smooth = False
        b += ln
        b.width, b.height = 26, 10
        estilo_ejes(b)
        nota(w, rt2 + 2,
             "CÓMO LEER EL GRÁFICO: las barras son los lotes que se producen cada mes; la línea gris es la demanda real; "
             "la línea ámbar es el inventario final planificado. Si la barra no alcanza a cubrir la línea gris y el inventario "
             "se agota, aparece el faltante (celda roja).", "L", alto=48, color=GR)
        w.add_chart(b, f"A{rt2 + 3}")

    hoja_pmp("PMP_Con_IA", t_ia, VE, "Con IA (MLP)", "el pronóstico de la red neuronal (MLP)")
    hoja_pmp("PMP_Sin_IA", t_sin, AZ, "Sin IA (Regresión)", "el pronóstico de la regresión desestacionalizada")

    # ================= HOJA 5: EXPLICACIÓN =================
    we = wb.create_sheet("Explicacion")
    we.sheet_properties.tabColor = "64748B"
    titulo(we, "Explicación del método, fórmulas y glosario", "B")
    filas = [
        ("OBJETIVO", "Comparar el costo y la calidad del pronóstico al trabajar SIN IA (regresión desestacionalizada) "
                     "frente a CON IA (red neuronal MLP) y ver cómo cada uno afecta al Plan Maestro de Producción."),
        ("DATOS", "CSV con Periodo, Mes_Num, Mes, Demanda_Real y Pedidos. Los primeros N − 12 periodos entrenan los modelos; "
                  "los últimos 12 se usan para evaluar (el modelo no los ve al entrenar)."),
        ("MÉTODO TRADICIONAL (Sin IA)", "1) Índice estacional de cada mes = promedio de ese mes ÷ promedio general. "
                                        "2) Demanda desestacionalizada = Demanda ÷ índice. "
                                        "3) Regresión lineal y = a·Periodo + b sobre la demanda desestacionalizada. "
                                        "4) Pronóstico = (a·Periodo + b) × índice del mes."),
        ("MÉTODO CON IA (MLP)", "Pipeline de scikit-learn: StandardScaler + MLPRegressor (capas 64 y 32, activación ReLU, "
                                "solver lbfgs, alpha = 1.0, max_iter = 2000, random_state = 42). Variables: Periodo, "
                                "sen/cos(2π·mes/12), sen/cos(4π·mes/12) y las interacciones Periodo × cada una. "
                                "Seno y coseno hacen que diciembre quede «cerca» de enero; las interacciones permiten que "
                                "la altura de los picos cambie con el tiempo."),
        ("PLAN MAESTRO (MPS / PMP)", "Requerimiento = máx(Pronóstico, Pedidos). Si Inv. Inicial < Requerimiento: "
                                     "MPS = ⌈(Requerimiento − Inv. Inicial) ÷ Lote⌉ × Lote; si no, 0. "
                                     "Inv. Final = Inv. Inicial + MPS − Requerimiento (y es el Inv. Inicial del mes siguiente)."),
        ("DPP (Disponible Para Promesa)", "Solo en meses con MPS > 0: Inv. Inicial + MPS − suma de Pedidos desde ese mes "
                                          "hasta el mes anterior al próximo lote. En los demás meses se muestra «-». "
                                          "Indica cuántas unidades se pueden comprometer con nuevos clientes."),
        ("EVALUACIÓN DE COSTOS", "Se simula mes a mes contra la Demanda Real: Disponible = Inv. Real + MPS. "
                                 "Si Disponible ≥ Demanda: Inv. Real = Disponible − Demanda y Faltante = 0; si no: "
                                 "Faltante = Demanda − Disponible, Inv. Real = 0 y se cuenta 1 quiebre. "
                                 "Costo total = Σ(Inv. Real × costo almacenamiento + Faltante × costo por faltante)."),
        ("MAPE", "Error Porcentual Absoluto Medio = promedio de |Real − Pronóstico| ÷ Real. Se expresa en %."),
        ("MAD", "Desviación Absoluta Media = promedio de |Real − Pronóstico|. Se expresa en unidades."),
        ("KPI", "Indicador clave de desempeño. Aquí: MAPE, MAD, costo total operativo y periodos con quiebre de stock."),
        ("QUIEBRE DE STOCK", "Mes en que la demanda real supera lo disponible (inventario + producción) y queda demanda sin atender."),
        ("MLP / ReLU / StandardScaler", "MLP: red neuronal de varias capas. ReLU: función de activación max(0, x) que permite "
                                        "aprender curvas. StandardScaler: normaliza cada variable (x − media) ÷ desviación."),
        ("PLAN AGREGADO vs. PMP", "El plan agregado planifica por familias y meses (capacidad, mano de obra, inventario); "
                                  "el PMP lo baja a productos y periodos concretos; luego el MRP calcula los materiales."),
    ]
    we["A3"], we["B3"] = "Concepto", "Explicación"
    hdr(we["A3"])
    hdr(we["B3"])
    for k, (a, b_) in enumerate(filas):
        r = 4 + k
        celda(we, f"A{r}", a, negrita=True, fondo=CE)
        celda(we, f"B{r}", b_)
        we[f"A{r}"].alignment = Alignment(wrap_text=True, vertical="top")
        we[f"B{r}"].alignment = Alignment(wrap_text=True, vertical="top")
        we.row_dimensions[r].height = max(30, 16 * (len(b_) // 105 + 1) + 6)
    we.column_dimensions["A"].width = 32
    we.column_dimensions["B"].width = 115

    wb.save(ruta)


def exportar(formato, estado):
    if not estado:
        raise gr.Error("Primero ejecuta el análisis en la pestaña de Resumen.")
    carpeta = tempfile.mkdtemp()
    t_sin = estado["res_sin"]["tabla"]
    t_ia = estado["res_ia"]["tabla"]
    if formato == "Excel (.xlsx)":
        ruta = os.path.join(carpeta, "PMP_Reporte.xlsx")
        construir_excel(ruta, estado)
    else:
        ruta = os.path.join(carpeta, "PMP_Reporte.csv")
        out = pd.concat([t_sin.assign(Escenario="Sin IA"), t_ia.assign(Escenario="Con IA")])
        out.to_csv(ruta, index=False, encoding="utf-8-sig")
    boton = gr.DownloadButton(label=f"⬇️ DESCARGAR {os.path.basename(ruta)}", value=ruta, visible=True)
    mensaje = "✅ **Archivo generado.** Presiona el botón verde de abajo para descargarlo."
    return boton, mensaje


# ---------------------------------------------------------------------
# 8. INTERFAZ (tema claro forzado)
# ---------------------------------------------------------------------
tema = gr.themes.Soft(
    primary_hue="blue", secondary_hue="emerald", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "Segoe UI", "Arial", "sans-serif"],
).set(
    body_background_fill=CELESTE, body_background_fill_dark=CELESTE,
    background_fill_primary=BLANCO, background_fill_primary_dark=BLANCO,
    background_fill_secondary=CELESTE, background_fill_secondary_dark=CELESTE,
    block_background_fill=BLANCO, block_background_fill_dark=BLANCO,
    block_border_color=BORDE, block_border_color_dark=BORDE,
    block_label_background_fill=BLANCO, block_label_background_fill_dark=BLANCO,
    block_title_text_color=TEXTO, block_title_text_color_dark=TEXTO,
    body_text_color=TEXTO, body_text_color_dark=TEXTO,
    body_text_color_subdued="#475569", body_text_color_subdued_dark="#475569",
    input_background_fill=BLANCO, input_background_fill_dark=BLANCO,
    border_color_primary="#E2E8F0", border_color_primary_dark="#E2E8F0",
    table_even_background_fill=BLANCO, table_even_background_fill_dark=BLANCO,
    table_odd_background_fill=FONDO, table_odd_background_fill_dark=FONDO,
    button_primary_background_fill=AZUL, button_primary_background_fill_dark=AZUL,
    button_primary_background_fill_hover="#1D4ED8",
    button_primary_background_fill_hover_dark="#1D4ED8",
    button_primary_text_color="#FFFFFF", button_primary_text_color_dark="#FFFFFF",
)

CSS = f"""
:root, .dark, body, .gradio-container {{
    color-scheme: light !important;
    background: {CELESTE} !important;
    color: {TEXTO};
}}
/* Bloques blancos con borde y sombra para que se distingan del fondo celeste */
.block, .form, .gr-group, .gradio-plot, .gradio-dataframe, .table-wrap {{
    background: {BLANCO} !important;
    border: 1px solid {BORDE} !important;
    border-radius: 14px !important;
    box-shadow: 0 3px 12px rgba(37,99,235,.12) !important;
}}
.tabitem, div[role="tabpanel"] {{ background: transparent !important; border: none !important;
    box-shadow: none !important; }}
.tab-nav, div[role="tablist"] {{ background: transparent !important; }}
/* Ocultar barra de herramientas de Plotly (se encimaba con la leyenda) */
.modebar-container, .modebar {{ display: none !important; }}
.js-plotly-plot, .plot-container, .plotly {{ width: 100% !important; }}
.gradio-container {{ max-width: 1280px !important; margin: auto; }}
#banner {{
    background: linear-gradient(135deg, {AZUL} 0%, #1D4ED8 55%, {VERDE} 140%);
    color: #FFFFFF; border-radius: 16px; padding: 22px 26px; margin-bottom: 12px;
    box-shadow: 0 6px 18px rgba(37,99,235,.25);
}}
#banner h1 {{ margin: 0 0 6px 0; font-size: 26px; font-weight: 800; color: #FFFFFF; }}
#banner p {{ margin: 0; opacity: .95; font-size: 14.5px; color: #FFFFFF; }}
#btn-run {{ background: {VERDE} !important; border: none !important; color: #FFFFFF !important;
           font-weight: 700; font-size: 16px; }}
#btn-run:hover {{ background: #059669 !important; }}
#btn-desc {{ background: {VERDE} !important; border: none !important; color: #FFFFFF !important;
            font-weight: 800; font-size: 17px; padding: 14px 0 !important;
            box-shadow: 0 4px 14px rgba(16,185,129,.45) !important; }}
#btn-desc:hover {{ background: #059669 !important; }}
.tab-nav button.selected, button[role="tab"][aria-selected="true"] {{
    color: {AZUL} !important; border-color: {AZUL} !important; font-weight: 700;
}}
"""

JS_CLARO = """
() => {
    const forzar = () => {
        document.body.classList.remove('dark');
        document.documentElement.classList.remove('dark');
        document.documentElement.style.colorScheme = 'light';
    };
    forzar();
    new MutationObserver(forzar).observe(document.body, {attributes: true, attributeFilter: ['class']});
}
"""

with gr.Blocks(theme=tema, css=CSS, js=JS_CLARO,
               title="SIS-310 · Pronóstico con IA y Plan Maestro") as demo:

    estado = gr.State(None)

    gr.HTML("""
    <div id="banner">
      <h1>📈 Pronóstico con IA y Plan Maestro de Producción</h1>
      <p>SIS-310 Sistemas de Producción · Regresión desestacionalizada vs. Red Neuronal (MLP)
         · PMP / DPP · KPIs de costos y quiebres de stock</p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("### 1️⃣ Datos de entrada")
            archivo = gr.File(label="Historial de demanda (.csv)", file_types=[".csv"],
                              type="filepath")
            gr.Markdown(
                "<small>Columnas: <code>Periodo, Mes_Num, Mes, Demanda_Real, Pedidos</code>. "
                "Los últimos 12 periodos se usan para evaluar.</small>")
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("### 2️⃣ Parámetros del plan maestro")
            inv0 = gr.Slider(0, 3000, value=250, step=10, label="Inventario inicial (unidades)")
            lote = gr.Slider(50, 3000, value=500, step=50, label="Tamaño de lote fijo MPS (unidades)")
            with gr.Row():
                c_alm = gr.Number(value=5, label="Costo almacenamiento (Bs./unidad)", minimum=0)
                c_falt = gr.Number(value=65, label="Costo por faltante (Bs./unidad)", minimum=0)
    btn = gr.Button("🚀 Analizar y calcular Plan Maestro", elem_id="btn-run", size="lg")

    with gr.Tabs():
        with gr.Tab("📊 Resumen Ejecutivo y Gráficos"):
            kpis = gr.HTML()
            gr.Markdown("#### Comparación Sin IA vs. Con IA")
            tabla_comp = gr.Dataframe(interactive=False, wrap=True)
            g_pron = gr.Plot(show_label=False)
            with gr.Row():
                g_err = gr.Plot(show_label=False)
            g_kpi = gr.Plot(show_label=False)

        with gr.Tab("🏭 Plan Maestro de Producción (MPS / PMP)"):
            escenario = gr.Radio(["Con IA (MLP)", "Sin IA (Regresión)"], value="Con IA (MLP)",
                                 label="Escenario a visualizar")
            tabla_pmp = gr.Dataframe(interactive=False, wrap=True, label="PMP periodo a periodo")
            g_pmp = gr.Plot(show_label=False)
            gr.Markdown(
                "<small><b>Requerimiento</b> = máx(Pronóstico, Pedidos). "
                "<b>DPP</b> solo se muestra en periodos con producción. "
                "<b>Inv. Real / Faltante</b> provienen de simular contra la demanda real.</small>")

        with gr.Tab("📥 Exportar Reporte"):
            gr.Markdown(
                "### 📥 Exportar el reporte\n"
                "**Paso 1:** elige el formato.  **Paso 2:** presiona *Generar archivo*.  "
                "**Paso 3:** presiona el botón verde *DESCARGAR* que aparecerá.\n\n"
                "El **Excel** incluye hojas con gráficos, fórmulas vivas y la explicación de cada tabla; "
                "el CSV trae solo las tablas del PMP.")
            formato = gr.Radio(["Excel (.xlsx)", "CSV (.csv)"], value="Excel (.xlsx)",
                               label="Paso 1 · Formato")
            btn_exp = gr.Button("📄 Paso 2 · Generar archivo", variant="primary")
            msg_exp = gr.Markdown()
            btn_desc = gr.DownloadButton("⬇️ Descargar", visible=False, size="lg",
                                         elem_id="btn-desc")

    btn.click(
        ejecutar, inputs=[archivo, inv0, lote, c_alm, c_falt],
        outputs=[kpis, tabla_comp, g_pron, g_err, g_kpi, tabla_pmp, g_pmp, estado])
    escenario.change(mostrar_pmp, inputs=[escenario, estado], outputs=[tabla_pmp, g_pmp])
    btn_exp.click(exportar, inputs=[formato, estado], outputs=[btn_desc, msg_exp])

demo.queue()
demo.launch(share=True, debug=False)

/tmp/ipykernel_3721/487124486.py:906: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=tema, css=CSS, js=JS_CLARO,
/tmp/ipykernel_3721/487124486.py:906: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=tema, css=CSS, js=JS_CLARO,
/tmp/ipykernel_3721/487124486.py:906: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=tema, css=CSS, js=JS_CLARO,


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d376d1799fe334b9d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
